# **Coffee Cafe - Statistic Analysis**

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlalchemy as sql
import urllib
import re

import pandas as pd
import numpy as np

from scipy.stats import (chi2_contingency, ttest_ind, f_oneway, pearsonr, levene)

## **Data Loading**

In [2]:
# Connection parameters
server = 'DESKTOP-3QCNPJH\RAYN'
database = 'Practice'

params = urllib.parse.quote_plus(
    f'DRIVER=ODBC Driver 17 for SQL Server;'
    f'SERVER={server};'
    f'DATABASE={database};'
    'Trusted_Connection=yes;'
)

engine = sql.create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

# Test connection
print(engine)

Engine(mssql+pyodbc:///?odbc_connect=DRIVER%3DODBC+Driver+17+for+SQL+Server%3BSERVER%3DDESKTOP-3QCNPJH%5CRAYN%3BDATABASE%3DPractice%3BTrusted_Connection%3Dyes%3B)


<>:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<>:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_10896\2215312596.py:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
  server = 'DESKTOP-3QCNPJH\RAYN'


In [3]:
customers_df     = pd.read_sql("SELECT * FROM cl_customers", engine)
order_details_df = pd.read_sql("SELECT * FROM cl_order_details", engine)
orders_df        = pd.read_sql("SELECT * FROM cl_orders", engine)
products_df      = pd.read_sql("SELECT * FROM cl_products", engine)

## **Data Cleaning**

In [4]:
orders_df['order_date']    = pd.to_datetime(orders_df['order_date'], format = '%d-%m-%Y')
customers_df['birth_date'] = pd.to_datetime(customers_df['birth_date'], format = '%d-%m-%Y')

orders_df = pd.merge(
                orders_df, 
                order_details_df.groupby('order_id')['line_total'].sum(), 
                on  = 'order_id', 
                how = 'inner'
)


**Customer + Order**

In [5]:
customers_order_df = pd.merge(
                        left  = orders_df,
                        right = customers_df[['customer_id', 'birth_date', 'customer_segment', 'gender']], 
                        on    = 'customer_id', 
                        how   = 'left'
)

customers_order_df['age_at_order'] = (
                (customers_order_df['order_date'] - customers_order_df['birth_date']).dt.days / 365.25
).astype(int)

customers_order_df['age_group'] = pd.cut(
                                            customers_order_df['age_at_order'],
                                            bins   = [16, 24, 32, 40, 48, 56, 63],
                                            labels = ['16-23','24-31','32-39','40-47','48-55','56-63']
)

**Products + Order Details**

In [6]:
product_order_df = pd.merge(
                        left  = order_details_df,
                        right = products_df[['product_id', 'category', 'selling_price']], 
                        on    = 'product_id', 
                        how   = 'left'
)

product_order_df

,detail_id,order_id,product_id,quantity,unit_price,discount_rate,line_total,ProductOrderNumber,category,selling_price
0,DTL003307,ORD00863,PRD001,3,39865,0.000000,119595,1,Coffee,39865
1,DTL003309,ORD00863,PRD003,2,26835,0.000000,53670,1,Coffee,26835
2,DTL001765,ORD00457,PRD009,3,19612,0.000000,58836,1,Coffee,19612
3,DTL001767,ORD00457,PRD021,1,22179,0.000000,22179,1,Tea,22179
4,DTL001764,ORD00457,PRD021,1,22179,0.099113,19980,2,Tea,22179
...,...,...,...,...,...,...,...,...,...,...
3757,DTL001012,ORD00263,PRD021,1,22179,0.000000,22179,1,Tea,22179
3758,DTL001893,ORD00490,PRD033,4,31779,0.000000,127116,1,Rice Bowl,31779
3759,DTL001892,ORD00490,PRD043,1,33409,0.176001,27528,1,Noodles,33409
3760,DTL001009,ORD00263,PRD060,1,27254,0.224679,21130,1,Snack,27254


## **Inference**

In [7]:
report_df = pd.DataFrame(columns = ['hyp_id', 'method', 'kasus', 'stat', 'p_value', 'keputusan', 'kesimpulan_1', 'effect_size', 'kesimpulan_2'])

### **One Way ANOVA & Eta-Squared**

#### **Customer Age vs Sales**

Tujuan

> Mengetahui apakah terdapat perbedaan rata-rata sales berdasarkan kelompok usia customer.

Variabel

- Independent variable $\to$ Customer Age Group
- Dependent variable: Total Sales

Hipotesis

- H₀ $\to$ Rata-rata sales seluruh kelompok usia sama.
- H₁ $\to$ Setidaknya terdapat satu kelompok usia dengan rata-rata sales yang berbeda.

Business Insight

> Hasil analisis dapat membantu bisnis mengetahui apakah kelompok usia tertentu memiliki kontribusi sales yang berbeda sehingga strategi pemasaran atau penawaran produk dapat disesuaikan.

**Hypothesis Testing**

In [8]:
h1_split = {}

for agegr in customers_order_df.age_group.unique() :
    h1_split[agegr] = customers_order_df[customers_order_df['age_group'] == agegr]['line_total'].values
    
h1_groups = [age_group for age_group in h1_split.values()]

In [9]:
h1_f_stat, h1_p_value = f_oneway(*h1_groups)

if h1_p_value < 0.05:
    
    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h1_keputusan = 'Tolak H0'
    h1_kesimpulan = 'Terdapat perbedaan Total Sales yang signifikan antar kelompok usia'
    
    # --- Menghitung Effect Size
    
    # Grand Mean
    h1_all_data = pd.concat(h1_groups)
    h1_grand_mean = h1_all_data.mean()

    # Sum of Squares Between
    h1_ss_between = sum(len(group) * (group.mean() - h1_grand_mean) ** 2 for group in h1_groups)

    # Total Sum of Squares
    h1_ss_total = sum((x - h1_grand_mean) ** 2 for x in h1_all_data)

    # Eta Squared
    h1_eta_squared = h1_ss_between / h1_ss_total
    
    if h1_eta_squared < 0.01:
        h1_effect = "Negligible"
    
    elif h1_eta_squared < 0.06:
        h1_effect = "Small"
    
    elif h1_eta_squared < 0.14:
        h1_effect = "Medium"
    
    else:
        h1_effect = "Large"
    
    h1_effect_size = np.round(h1_eta_squared, 3)
    h1_kesimpulan_2 = f"Terdapat perbedaan Total Sales antar kelompok usia dengan effect size {h1_effect}"

else:
    h1_keputusan = 'Gagal Tolak H0'
    h1_kesimpulan = 'Tidak terdapat perbedaan Total Sales yang signifikan antar kelompok usia'
    h1_effect_size = '-'
    h1_kesimpulan_2 = h1_kesimpulan

**Report**

In [10]:
report_df.loc[0, :] = [
                        'h1', 
                        'One Way ANOVA',
                        'Cust Age vs Total Sales', 
                        h1_f_stat, 
                        h1_p_value, 
                        h1_keputusan, 
                        h1_kesimpulan, 
                        h1_effect_size, 
                        h1_kesimpulan_2
]

#### **Product Category vs Sales**

Tujuan

> Mengetahui apakah rata-rata sales berbeda antar kategori produk.

Variabel

- Independent variable $\to$ Product Category
- Dependent variable $\to$ Sales

Hipotesis

- H₀: Rata-rata sales seluruh kategori produk sama.
- H₁: Setidaknya terdapat satu kategori dengan rata-rata sales yang berbeda.

Business Insight

> Hasil ini dapat digunakan untuk menentukan kategori produk mana yang memiliki performa berbeda secara signifikan dan membutuhkan perhatian lebih lanjut dalam strategi produk, inventory, maupun pemasaran.

**Hypothesis Testing**

In [11]:
h2_split = {}

for prod_cat in product_order_df['category'].unique() :
    h2_split[prod_cat] = product_order_df[product_order_df['category'] == prod_cat]['line_total'].values

h2_groups = [prod_cat for prod_cat in h2_split.values()]

In [12]:
h2_f_stat, h2_p_value = f_oneway(*h2_groups)

if h2_p_value < 0.05:
    
    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h2_keputusan = 'Tolak H0'
    h2_kesimpulan = 'Terdapat perbedaan Total Sales yang signifikan antar product category'
    
    # --- Menghitung Effect Size
    
    # Grand Mean
    h2_all_data = []
    
    for group in range(len(h2_groups)) :
        for val in range(len(h2_groups[group])) :
            h2_all_data.append(h2_groups[group][val])
            
    h2_grand_mean = np.mean(h2_all_data)

    # Sum of Squares Between
    h2_ss_between = sum(len(group) * (group.mean() - h2_grand_mean) ** 2 for group in h2_groups)

    # Total Sum of Squares
    h2_ss_total = sum((x - h2_grand_mean) ** 2 for x in h2_all_data)

    # Eta Squared
    h2_eta_squared = h2_ss_between / h2_ss_total
    
    if h2_eta_squared < 0.01:
        h2_effect = "Negligible"
        
    elif h2_eta_squared < 0.06:
        h2_effect = "Small"
        
    elif h2_eta_squared < 0.14:
        h2_effect = "Medium"
        
    else:
        h2_effect = "Large"
    
    h2_effect_size = np.round(h2_eta_squared, 3)
    h2_kesimpulan_2 = f"Terdapat perbedaan Total Sales antar kelompok kategori produk dengan effect size {h2_effect}"

else:
    h2_keputusan = 'Gagal Tolak H0'
    h2_kesimpulan = 'Tidak terdapat perbedaan Total Sales yang signifikan antar kategori produk'
    h2_effect_size = '-'
    h2_kesimpulan_2 = h2_kesimpulan

**Report**

In [13]:
report_df.loc[1, :] = ['h2', 'One Way ANOVA','Prod Category vs Total Sales', h2_f_stat, h2_p_value, h2_keputusan, h2_kesimpulan, h2_effect_size, h2_kesimpulan_2]

### **Independent T-test & Cohens'd**

#### **Weekend vs Sales**

Tujuan

> Mengetahui apakah rata-rata sales berbeda antara weekday dan weekend.

Variabel

- Group $\to$ Weekday vs Weekend
- Metric $\to$ Total Sales

Hipotesis

- H₀ $\to$ Rata-rata sales weekday sama dengan weekend.
- H₁ $\to$ Rata-rata sales weekday berbeda dengan weekend.
  
Business Insight

> Hasil ini dapat membantu menentukan apakah periode weekend membutuhkan strategi operasional, inventory, staffing, atau promosi yang berbeda dibandingkan weekday.

**Hypothesis Testing**

In [14]:
customers_order_df['day_of_week'] = customers_order_df['order_date'].dt.dayofweek
customers_order_df['is_weekend']  = customers_order_df['day_of_week'].apply(lambda x : 1 if x in [0, 6] else 0)

In [15]:
h3_split = {}

for weekend_status in customers_order_df.is_weekend.unique() :
    h3_split[weekend_status] = customers_order_df[customers_order_df['is_weekend'] == weekend_status]['line_total'].values

h3_groups = [weekend_status for weekend_status in h3_split.values()]

In [16]:
h3_t_stat, h3_p_value = ttest_ind(*h3_groups)

if h3_p_value < 0.05:

    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h3_keputusan = 'Tolak H0'
    h3_kesimpulan = 'Terdapat perbedaan yang signifikan pada total sales antara waktu weekend maupun tidak'
    
    # --- Menghitung Effect Size
    h3_n1 = len(h3_groups[0])
    h3_n2 = len(h3_groups[1])

    h3_mean1 = h3_groups[0].mean()
    h3_mean2 = h3_groups[1].mean()

    sd1 = h3_groups[0].std(ddof=1)
    sd2 = h3_groups[1].std(ddof=1)

    # Pooled Standard Deviation
    h3_pooled_sd = np.sqrt(((h3_n1 - 1) * sd1 ** 2 + (h3_n2 - 1) * sd2 ** 2) / (h3_n1 + h3_n2 - 2))
    
    h3_cohens_d = (h3_mean1 - h3_mean2) / h3_pooled_sd
    h3_abs_cohens_d = np.abs(h3_cohens_d)

    if h3_abs_cohens_d < 0.20:
        h3_effect = "Negligible"
        
    elif h3_abs_cohens_d < 0.50:
        h3_effect = "Small"
        
    elif h3_abs_cohens_d < 0.80:
        h3_effect = "Medium"
        
    else:
        h3_effect = "Large"
    
    h3_effect_size  = np.round(h3_cohens_d)
    h3_kesimpulan_2 = h3_effect
    
else:
    h3_keputusan = 'Gagal Tolak H0'
    h3_kesimpulan = 'Tidak terdapat perbedaan yang signifikan pada total sales antara waktu weekend atau tidak'
    h3_effect_size = '-'
    h3_kesimpulan_2 = h3_kesimpulan

**Report**

In [17]:
report_df.loc[2, :] = ['h3', 'Independent T-test', 'Weekend vs Total Sales', h3_t_stat, h3_p_value, h3_keputusan, h3_kesimpulan, h3_effect_size, h3_kesimpulan_2]

#### **Customer Gender vs Sales**

Tujuan

> Mengetahui apakah rata-rata sales berbeda berdasarkan gender customer.

Variabel

- Group $\to$ Gender
- Metric $\to$ Total Sales

Hipotesis

- H₀ $\to$ Rata-rata sales antara kedua kelompok gender sama.
- H₁ $\to$ Rata-rata sales antara kedua kelompok gender berbeda.

Business Insight

- Analisis ini dapat digunakan sebagai exploratory analysis untuk mengetahui apakah terdapat pola perbedaan perilaku pembelian berdasarkan gender.
- Hasilnya sebaiknya digunakan sebagai dasar eksplorasi lebih lanjut, bukan sebagai alasan untuk membuat keputusan pemasaran yang terlalu general.

**Hypothesis Testing**

In [18]:
h4_split = {}

for gender_type in customers_order_df.gender.unique() :
    h4_split[gender_type] = customers_order_df[customers_order_df['gender'] == gender_type]['line_total'].values
    
h4_groups = [gender for gender in h4_split.values()]

In [19]:
h4_t_stat, h4_p_value = ttest_ind(*h4_groups)

if h4_p_value < 0.05:

    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h4_keputusan = 'Tolak H0'
    h4_kesimpulan = 'Terdapat perbedaan yang signifikan pada total sales antar gender'
    
    # --- Menghitung Effect Size
    h4_n1 = len(h4_groups[0])
    h4_n2 = len(h4_groups[1])

    h4_mean1 = h4_groups[0].mean()
    h4_mean2 = h4_groups[1].mean()

    sd1 = h4_groups[0].std(ddof=1)
    sd2 = h4_groups[1].std(ddof=1)

    # Pooled Standard Deviation
    h4_pooled_sd = np.sqrt(((h4_n1 - 1) * sd1 ** 2 + (h4_n2 - 1) * sd2 ** 2) / (h4_n1 + h4_n2 - 2))
    
    h4_cohens_d = (h4_mean1 - h4_mean2) / h4_pooled_sd
    h4_abs_cohens_d = np.abs(h4_cohens_d)

    if h4_abs_cohens_d < 0.20:
        h4_effect = "Negligible"
        
    elif h4_abs_cohens_d < 0.50:
        h4_effect = "Small"
        
    elif h4_abs_cohens_d < 0.80:
        h4_effect = "Medium"
        
    else:
        h4_effect = "Large"
    
    h4_effect_size = np.round(h4_cohens_d, 3)
    h4_kesimpulan_2 = h4_effect
    
else:
    h4_keputusan = 'Gagal Tolak H0'
    h4_kesimpulan = 'Tidak terdapat perbedaan yang signifikan pada total sales antar gender'
    h4_effect_size = '-'
    h4_kesimpulan_2 = h4_kesimpulan

**Report**

In [20]:
report_df.loc[3, :] = ['h4', 'Independent T-test', 'Cust Gender vs Total Sales', h4_t_stat, h4_p_value, h4_keputusan, h4_kesimpulan, h4_effect_size, h4_kesimpulan_2]

### **Chi-Square & Crammer's V**

In [21]:
customer_order_segment = customers_order_df[['order_id', 'customer_segment', 'payment_method']]
customer_order_details = pd.merge(
                            left  = order_details_df[['order_id', 'quantity']],
                            right = customer_order_segment, 
                            on    = 'order_id', 
                            how   = 'left'
)

quantity_per_order = customer_order_details.groupby('order_id').aggregate({'quantity' : 'sum', 'customer_segment' : 'max', 'payment_method' : 'max'})
quantity_per_order['quantity'] = quantity_per_order['quantity'].apply(lambda x : 'small' if x <= 2 else "Medium" if x <= 4 else "Large")

#### **Customer Segment vs Order Size**

Tujuan

> Mengetahui apakah customer segment memiliki hubungan dengan ukuran basket atau jumlah item dalam satu order.

Variabel

- Variable 1 $\to$ Customer Segment
- Variable 2 $\to$ Items/Order Category

Contoh kategori:

- Small → 1–2 items
- Medium → 3–4 items
- Large → 5+ items

Hipotesis

- H₀ $\to$ Customer segment dan items/order bersifat independen.
- H₁ $\to$ Customer segment dan items/order memiliki hubungan.

**Hypothesis Testing**

In [22]:
# --- Mempersiapkan Data
customer_segment_order_size = quantity_per_order[['quantity', 'customer_segment']]

# --- Membuat Tabel Contingency
h5_contingency = customer_segment_order_size.pivot_table(index = 'customer_segment', columns = 'quantity', aggfunc = 'size')

# --- Melakukan Uji Hypotesis Chi Square
h5_chi2, h5_p_value, h5_dof, h5_expected = chi2_contingency(h5_contingency)

alpha = 0.05

if h5_p_value < alpha:
    
    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h5_keputusan = "Tolak H0"
    h5_kesimpulan = "Terdapat hubungan yang signifikan antara customer segment dengan jumlah order size"
    
    # --- Menghitung effect size
    n5 = h5_contingency.to_numpy().sum()
    h5_rows, h5_cols = h5_contingency.shape
    
    k5 = min(h5_rows, h5_cols)

    h5_cramers_v = np.sqrt(h5_chi2 / (n5 * (k5 - 1)))
    
    if h5_cramers_v < 0.10:
        h5_effect = "Negligible"
        
    elif h5_cramers_v < 0.30:
        h5_effect = "Small"
        
    elif h5_cramers_v < 0.50:
        h5_effect = "Medium"
        
    else:
        h5_effect = "Large"

    h5_effect_size = np.round(h5_cramers_v,3)
    h5_kesimpulan_2 = f"Segmen pelanggan memiliki hubungan yang {h5_effect} dengan jumlah order size"
    
else:
    h5_keputusan = "Gagal Tolak H0"
    h5_kesimpulan = "Tidak terdapat bukti hubungan yang signifikan antara customer segment dengan jumlah order size"
    h5_effect_size = '-'
    h5_kesimpulan_2 = h5_kesimpulan

**Report**

In [23]:
report_df.loc[4, :] = ['h5', 'Chi Square', 'Cust Segment vs Order Size', h5_chi2, h5_p_value, h5_keputusan, h5_kesimpulan, h5_effect_size, h5_kesimpulan_2]

#### **Payment Method vs Order Size**

Tujuan

> Mengetahui apakah metode pembayaran memiliki hubungan dengan ukuran basket customer.

Variabel

- Variable 1 $\to$ Payment Method
- Variable 2 $\to$ Items/Order Category

Hipotesis

- H₀ $\to$ Payment method dan items/order tidak memiliki hubungan.
- H₁ $\to$ Payment method dan items/order memiliki hubungan.

Business Insight

> Analisis dapat memberikan indikasi apakah customer yang menggunakan metode pembayaran tertentu memiliki kecenderungan basket size yang berbeda.

**Hypothesis Testing**

In [24]:
# --- Mempersiapkan Data
payment_method_order_size = quantity_per_order[['quantity', 'payment_method']]

# --- Membuat Tabel Contingency
h6_contingency = payment_method_order_size.pivot_table(index = 'payment_method', columns = 'quantity', aggfunc = 'size')

# --- Melakukan Uji Hypotesis Chi Square
h6_chi2, h6_p_value, h6_dof, h6_expected = chi2_contingency(h6_contingency)

alpha = 0.05

if h6_p_value < alpha:

    # --- Mengambil keputusan dan kesimpulan berdasarkan hasis hipotesis testing
    h6_keputusan = "Tolak H0"
    h6_kesimpulan = "Terdapat hubungan yang signifikan antara payment method dengan jumlah order size"
    
    # --- Menghitung effect size
    n6 = h6_contingency.to_numpy().sum()
    h6_rows, h6_cols = h6_contingency.shape
    
    k6 = min(h6_rows, h6_cols)

    h6_cramers_v = np.sqrt(h6_chi2 / (n6 * (k6 - 1)))
    
    if h6_cramers_v < 0.10:
        h6_effect = "Negligible"
        
    elif h6_cramers_v < 0.30:
        h6_effect = "Small"
        
    elif h6_cramers_v < 0.50:
        h6_effect = "Medium"
        
    else:
        h6_effect = "Large"

    h6_effect_size = np.round(h6_cramers_v, 3)
    h6_kesimpulan_2 = f"Metode Pembayaran memiliki hubungan yang {h6_effect} dengan jumlah order size"
    
else:
    h6_keputusan = "Gagal Tolak H0"
    h6_kesimpulan = "Tidak terdapat bukti hubungan yang signifikan antara payment method dengan jumlah order size"
    h6_effect_size = '-'
    h6_kesimpulan_2 = h6_kesimpulan

**Report**

In [25]:
report_df.loc[5, :] = ['h6', 'Chi Square', 'Payment Method vs Order Size', h6_chi2, h6_p_value, h6_keputusan, h6_kesimpulan, h6_effect_size, h6_kesimpulan_2]

### **Pearson Correlation**

#### **Product Price vs Quantity**

Tujuan

> Mengetahui apakah harga produk memiliki hubungan linear dengan jumlah unit yang terjual.

Variabel

- X $\to$ Product Price
- Y $\to$ Quantity Sold

Hipotesis

- H₀ $\to$ Tidak terdapat hubungan linear antara product price dan quantity sold.
- H₁ $\to$ Terdapat hubungan linear antara product price dan quantity sold.

Business Insight

- Analisis ini dapat membantu mengevaluasi apakah harga produk memiliki hubungan dengan volume penjualan.
- Namun, korelasi tidak membuktikan bahwa harga menyebabkan perubahan quantity. Faktor lain seperti brand, category, quality, promotion, dan product popularity juga dapat memengaruhi penjualan.

In [26]:
h7_corr, h7_p_value = pearsonr(product_order_df['quantity'], product_order_df['unit_price'])

# Hypothesis Test
alpha = 0.05

if h7_p_value < alpha:
    h7_keputusan = "Tolak H0"
    h7_kesimpulan = "Terdapat korelasi yang signifikan antara harga product satuan dengan jumlah produk yang dibeli"
    
else:
    h7_keputusan = "Gagal Tolak H0"
    h7_kesimpulan = "Tidak terdapat korelasi yang signifikan antara harga product satuan dengan jumlah produk yang dibeli"

# --- Mencari Besar Kekuatan Korelasi

h7_abs_corr = abs(h7_corr)

if h7_abs_corr < 0.10:
    h7_corr_strength = "Negligible"
    
elif h7_abs_corr < 0.30:
    h7_corr_strength = "Weak"
    
elif h7_abs_corr < 0.50:
    h7_corr_strength = "Moderate"
    
elif h7_abs_corr < 0.70:
    h7_corr_strength = "Strong"
    
else:
    h7_corr_strength = "Very Strong"

# --- Mencari Arah Korelasi
if h7_corr > 0:
    h7_direction = "Positive"
    
elif h7_corr < 0:
    h7_direction = "Negative"
    
else:
    h7_direction = "No correlation"

h7_kesimpulan_2 = f"Product Price memiliki korelasi {h7_direction} yang {h7_corr_strength} dengan Total Quantity"

In [27]:
report_df.loc[6, :] = [
                        'h7', 
                        'Pearson Correlation',
                        'Prod Price vs Total Qty', 
                        h7_corr, 
                        h7_p_value, 
                        h7_keputusan, 
                        h7_kesimpulan, 
                        '-', 
                        h7_kesimpulan_2
]

#### **Item Sold vs Sales**

Tujuan

> Mengetahui apakah jumlah item dalam transaksi memiliki hubungan linear dengan total sales.

Variabel

- X $\to$ Items Sold / Quantity
- Y $\to$ Total Sales

Hipotesis

- H₀ $\to$ Tidak terdapat hubungan linear antara items sold dan sales.
- H₁ $\to$ Terdapat hubungan linear antara items sold dan sales.

Business Insight

> Hasil ini dapat membantu memahami seberapa erat volume item yang terjual berkaitan dengan pencapaian sales.

**Perhitungan**

In [28]:
h8_corr, h8_p_value = pearsonr(product_order_df['quantity'], product_order_df['line_total'])

# Hypothesis Test
alpha = 0.05

if h8_p_value < alpha:
    h8_keputusan = "Tolak H0"
    h8_kesimpulan = "Terdapat korelasi yang signifikan antara produk item sold dengan total penjualan"
    
else:
    h8_keputusan = "Gagal Tolak H0"
    h8_kesimpulan = "Tidak terdapat korelasi yang signifikan antara produk item sold dengan total penjualan"

# --- Mencari Besar Kekuatan Korelasi

h8_abs_corr = abs(h8_corr)

if h8_abs_corr < 0.10:
    h8_corr_strength = "Negligible"
    
elif h8_abs_corr < 0.30:
    h8_corr_strength = "Weak"
    
elif h8_abs_corr < 0.50:
    h8_corr_strength = "Moderate"
    
elif h8_abs_corr < 0.70:
    h8_corr_strength = "Strong"
    
else:
    h8_corr_strength = "Very Strong"

# --- Mencari Arah Korelasi
if h8_corr > 0:
    h8_direction = "Positive"
    
elif h8_corr < 0:
    h8_direction = "Negative"
    
else:
    h8_direction = "No correlation"

h8_kesimpulan_2 = f"Item Sold memiliki korelasi {h8_direction} yang {h8_corr_strength} dengan Total Sales"

In [29]:
report_df.loc[7, :] = [
                        'h8', 
                        'Pearson Correlation',
                        'Item Sold vs Total Sales', 
                        h8_corr, 
                        h8_p_value, 
                        h8_keputusan, 
                        h8_kesimpulan, 
                        '-', 
                        h8_kesimpulan_2
]

In [30]:
report_df['kesimpulan_1'] = report_df['kesimpulan_1'].apply(lambda txt : txt.title())
report_df['kesimpulan_2'] = report_df['kesimpulan_2'].apply(lambda txt : txt.title())

In [31]:
report_df

,hyp_id,method,kasus,stat,p_value,keputusan,kesimpulan_1,effect_size,kesimpulan_2
0,h1,One Way ANOVA,Cust Age vs Total Sales,0.335863,0.853885,Gagal Tolak H0,Tidak Terdapat Perbedaan Total Sales Yang Sign...,-,Tidak Terdapat Perbedaan Total Sales Yang Sign...
1,h2,One Way ANOVA,Prod Category vs Total Sales,61.487889,0.0,Tolak H0,Terdapat Perbedaan Total Sales Yang Signifikan...,0.089,Terdapat Perbedaan Total Sales Antar Kelompok ...
2,h3,Independent T-test,Weekend vs Total Sales,0.336766,0.736364,Gagal Tolak H0,Tidak Terdapat Perbedaan Yang Signifikan Pada ...,-,Tidak Terdapat Perbedaan Yang Signifikan Pada ...
3,h4,Independent T-test,Cust Gender vs Total Sales,-0.157332,0.875015,Gagal Tolak H0,Tidak Terdapat Perbedaan Yang Signifikan Pada ...,-,Tidak Terdapat Perbedaan Yang Signifikan Pada ...
4,h5,Chi Square,Cust Segment vs Order Size,4.655456,0.32451,Gagal Tolak H0,Tidak Terdapat Bukti Hubungan Yang Signifikan ...,-,Tidak Terdapat Bukti Hubungan Yang Signifikan ...
5,h6,Chi Square,Payment Method vs Order Size,13.835386,0.03153,Tolak H0,Terdapat Hubungan Yang Signifikan Antara Payme...,0.083,Metode Pembayaran Memiliki Hubungan Yang Negli...
6,h7,Pearson Correlation,Prod Price vs Total Qty,-0.010335,0.526274,Gagal Tolak H0,Tidak Terdapat Korelasi Yang Signifikan Antara...,-,Product Price Memiliki Korelasi Negative Yang ...
7,h8,Pearson Correlation,Item Sold vs Total Sales,0.886386,0.0,Tolak H0,Terdapat Korelasi Yang Signifikan Antara Produ...,-,Item Sold Memiliki Korelasi Positive Yang Very...


In [32]:
report_df.to_csv('data/statistic_data.csv', index = False)